# IBM Mapping Auswertung

Dieses Notebook wertet `Data/ostschweiz_mapping_ibm.csv` aus.

Ziele:
- Welche HD-Wörter wurden mit der höchsten IBM-Wahrscheinlichkeit gemappt?
- Welche Mappings sind zusätzlich robust, also mit genug Support in mehreren Sätzen?
- Welche häufigen HD-Wörter wurden gar nicht gemappt?
- Welche Wörter sind mehrdeutig oder unsicher?

Wichtig: `IBM_Prob` allein kann seltene Wörter zu stark aussehen lassen. Für die Interpretation sind `Score`, `Sentence_Support` und `Expected_Count` meistens aussagekräftiger.

In [ ]:
import re
from collections import Counter

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

IBM_CSV = DATA_DIR / "ostschweiz_mapping_ibm.csv"
PMI_CSV = DATA_DIR / "ostschweiz_mapping_results.csv"
INPUT_CSV = DATA_DIR / "transcriptions_tenses.csv"

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

## Daten laden

In [ ]:
df_ibm = pd.read_csv(IBM_CSV)
df = pd.read_csv(INPUT_CSV)
df_ost = df[df["dialect_region"] == "Ostschweiz"].copy()

print(f"IBM-Kandidaten: {len(df_ibm):,}")
print(f"Ostschweiz-Sätze: {len(df_ost):,}")
display(df_ibm.head())

In [ ]:
def clean_hd(text: str) -> list[str]:
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.split()


hd_freq = Counter()
for sentence in df_ost["sentence"]:
    hd_freq.update(clean_hd(sentence))

df_hd_freq = (
    pd.DataFrame(hd_freq.items(), columns=["Hochdeutsch", "HD_Frequenz"])
    .sort_values("HD_Frequenz", ascending=False)
    .reset_index(drop=True)
)

print(f"HD-Vokabular im Ostschweiz-Korpus: {len(df_hd_freq):,}")
display(df_hd_freq.head(20))

## Höchste IBM-Wahrscheinlichkeit

Diese Tabelle beantwortet direkt: Welche Paare haben die höchste gelernte Wahrscheinlichkeit `P(IPA | HD)`?

Achtung: Bei sehr seltenen Wörtern kann `IBM_Prob` hoch sein, obwohl wenig Evidenz da ist. Deshalb zeigen wir daneben `Sentence_Support`, `Expected_Count` und `HD_Gesamt_Haeufigkeit`.

In [ ]:
top_by_prob = (
    df_ibm
    .sort_values(["IBM_Prob", "Sentence_Support", "Expected_Count"], ascending=[False, False, False])
    .reset_index(drop=True)
)

display(top_by_prob[[
    "Hochdeutsch", "IPA_Dialekt", "Rang", "IBM_Prob", "Expected_Count",
    "Hard_Count", "Sentence_Support", "Score", "HD_Gesamt_Haeufigkeit",
    "IPA_Gesamt_Haeufigkeit"
]].head(50))

## Robusteste Mappings nach Score

`Score = IBM_Prob * log(1 + Expected_Count)`.

Das ist für die Interpretation oft besser als `IBM_Prob`, weil es Wahrscheinlichkeit und Evidenz kombiniert.

In [ ]:
top_by_score = (
    df_ibm[df_ibm["Rang"] == 1]
    .sort_values(["Score", "Sentence_Support"], ascending=[False, False])
    .reset_index(drop=True)
)

display(top_by_score[[
    "Hochdeutsch", "IPA_Dialekt", "IBM_Prob", "Expected_Count",
    "Sentence_Support", "Score", "Avg_Position_Distance",
    "HD_Gesamt_Haeufigkeit", "IPA_Gesamt_Haeufigkeit"
]].head(60))

In [ ]:
plt.figure(figsize=(9, 5))
plot_df = top_by_score.head(25).sort_values("Score")
sns.barplot(data=plot_df, y="Hochdeutsch", x="Score", color="#4C78A8")
plt.title("Top 25 IBM-Mappings nach Score")
plt.xlabel("Score")
plt.ylabel("Hochdeutsch")
plt.tight_layout()

## High-Confidence-Mappings

Hier setzen wir strengere Schwellen für eine vorsichtige Liste:
- `IBM_Prob >= 0.7`
- `Sentence_Support >= 5`
- `Avg_Position_Distance <= 0.12`
- nur bester Kandidat pro HD-Wort (`Rang == 1`)

In [ ]:
high_conf = df_ibm[
    (df_ibm["Rang"] == 1)
    & (df_ibm["IBM_Prob"] >= 0.7)
    & (df_ibm["Sentence_Support"] >= 5)
    & (df_ibm["Avg_Position_Distance"] <= 0.12)
].sort_values(["Score", "Sentence_Support"], ascending=[False, False])

print(f"High-Confidence-Mappings: {len(high_conf):,}")
display(high_conf[[
    "Hochdeutsch", "IPA_Dialekt", "IBM_Prob", "Expected_Count",
    "Sentence_Support", "Score", "Avg_Position_Distance",
    "HD_Gesamt_Haeufigkeit"
]].head(100))

## Welche HD-Wörter wurden nicht gemappt?

Nicht gemappt bedeutet hier: Das HD-Wort kommt im Ostschweiz-Korpus vor, aber es gibt keinen IBM-Kandidaten in `ostschweiz_mapping_ibm.csv`.

Interessant sind besonders häufige nicht gemappte Wörter, weil dort eigentlich genug Evidenz vorhanden sein könnte.

In [ ]:
mapped_words = set(df_ibm["Hochdeutsch"])
df_unmapped = df_hd_freq[~df_hd_freq["Hochdeutsch"].isin(mapped_words)].copy()

print(f"Gemappte HD-Wörter: {len(mapped_words):,}")
print(f"Nicht gemappte HD-Wörter: {len(df_unmapped):,}")
display(df_unmapped.head(100))

In [ ]:
plt.figure(figsize=(8, 5))
plot_df = df_unmapped.head(30).sort_values("HD_Frequenz")
sns.barplot(data=plot_df, y="Hochdeutsch", x="HD_Frequenz", color="#F58518")
plt.title("Häufigste nicht gemappte HD-Wörter")
plt.xlabel("HD-Frequenz im Korpus")
plt.ylabel("Hochdeutsch")
plt.tight_layout()

## Nicht gemappte mittelhäufige Wörter

Für eure Forschungsfrage sind mittelhäufige Wörter besonders relevant. Die Grenzen hier sind frei anpassbar.

In [ ]:
MIN_FREQ = 5
MAX_FREQ = 50

df_unmapped_mid = df_unmapped[
    df_unmapped["HD_Frequenz"].between(MIN_FREQ, MAX_FREQ)
].sort_values("HD_Frequenz", ascending=False)

print(f"Nicht gemappte mittelhäufige Wörter ({MIN_FREQ}-{MAX_FREQ} Vorkommen): {len(df_unmapped_mid):,}")
display(df_unmapped_mid.head(100))

## Unsichere oder mehrdeutige Mappings

Wenn ein HD-Wort mehrere Kandidaten hat oder die beste Wahrscheinlichkeit niedrig ist, sollte man es manuell prüfen.

In [ ]:
candidate_counts = (
    df_ibm.groupby("Hochdeutsch")
    .size()
    .reset_index(name="Anzahl_Kandidaten")
)

df_ambiguous = (
    df_ibm.merge(candidate_counts, on="Hochdeutsch")
    .query("Anzahl_Kandidaten > 1")
    .sort_values(["Anzahl_Kandidaten", "Hochdeutsch", "Rang"], ascending=[False, True, True])
)

print(f"HD-Wörter mit mehreren IBM-Kandidaten: {df_ambiguous['Hochdeutsch'].nunique():,}")
display(df_ambiguous[[
    "Hochdeutsch", "IPA_Dialekt", "Rang", "IBM_Prob", "Expected_Count",
    "Sentence_Support", "Score", "Anzahl_Kandidaten"
]].head(100))

In [ ]:
low_conf = df_ibm[
    (df_ibm["Rang"] == 1)
    & (
        (df_ibm["IBM_Prob"] < 0.5)
        | (df_ibm["Sentence_Support"] < 5)
        | (df_ibm["Avg_Position_Distance"] > 0.12)
    )
].sort_values(["HD_Gesamt_Haeufigkeit", "Score"], ascending=[False, False])

print(f"Potentiell unsichere beste Mappings: {len(low_conf):,}")
display(low_conf[[
    "Hochdeutsch", "IPA_Dialekt", "IBM_Prob", "Expected_Count",
    "Sentence_Support", "Score", "Avg_Position_Distance",
    "HD_Gesamt_Haeufigkeit"
]].head(100))

## Vergleich mit bisherigem PMI-Mapping

Diese Zelle zeigt, welche HD-Wörter beide Methoden mappen und wo nur IBM oder nur PMI etwas findet.

In [ ]:
df_pmi = pd.read_csv(PMI_CSV)

ibm_best = df_ibm[df_ibm["Rang"] == 1].copy()
comparison = ibm_best.merge(
    df_pmi,
    on="Hochdeutsch",
    how="outer",
    suffixes=("_IBM", "_PMI"),
    indicator=True,
)

print(comparison["_merge"].value_counts())

display(comparison[[
    "Hochdeutsch", "IPA_Dialekt_IBM", "IBM_Prob", "Sentence_Support",
    "Score", "IPA_Dialekt_PMI", "Gemeinsame_Treffer", "Kokkurrenz_Rate",
    "PMI", "_merge"
]].sort_values(["_merge", "Score"], ascending=[True, False]).head(120))

## Kurze Interpretation für das Paper

Bausteine, die ihr in die Auswertung übernehmen könnt:

- `IBM_Prob` beschreibt, wie stark das Modell ein IPA-Token als Übersetzung/Entsprechung eines HD-Worts gelernt hat.
- `Sentence_Support` schützt vor Zufallstreffern, weil ein Mapping in mehreren Sätzen als beste harte Zuordnung auftreten muss.
- `Avg_Position_Distance` zeigt, ob das Mapping typischerweise an ähnlicher Satzposition liegt.
- Nicht gemappte mittelhäufige Wörter sind besonders interessant für Fehleranalyse: dort kann das Problem an Dialektvariation, ASR-Fehlern, zusammengesetzten Tokens oder echten n:m-Zuordnungen liegen.